# Lab 2 — Predicción de Resistencia a la Compresión

**Sesión:** 1 o 3 · **Tema:** Python, EDA y aprendizaje supervisado básico

**Público:** ingenieros civiles · **Entorno:** GitHub Codespaces o `labs/.venv`

## Cómo trabajar

1. Abre este repo en **Codespaces** o ejecuta `bash labs/setup.sh` y activa `source labs/.venv/bin/activate`.
2. Lee [`data/DATOS.md`](data/DATOS.md) para entender las variables del hormigón.
3. Ejecuta celdas **en orden**; solo modifica bloques `### TU TAREA AQUÍ ###`.
4. Tras cada pregunta, ejecuta **Autoevaluación** y busca ✅.
5. Referencia docente: `resistencia_compresion_solucion.ipynb`.

> El código de carga, gráficos y modelado está pre-escrito. Tu trabajo: **experimentar** con parámetros y **interpretar** resultados de obra.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from _verificar import (
    verificar_tipo_problema, verificar_carga, verificar_columna,
    verificar_resumen, verificar_umbral_resistencia, verificar_correlaciones,
    verificar_scatter_filtro, verificar_split, verificar_modelo,
    verificar_importancias, resumen_seccion,
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

%matplotlib inline
sns.set_theme(style='whitegrid')
print('✅ Entorno listo (Codespaces / labs/.venv).')


## Contexto del dataset (UCI)

| Variable | Unidad | En obra significa… |
|----------|--------|-------------------|
| Cemento, Escoria, CenizaVolante | kg/m³ | Dosificación de ligantes y adiciones |
| Agua | kg/m³ | Relación agua/cemento (W/C) |
| Superplastificante | kg/m³ | Reducir agua sin perder trabajabilidad |
| AgregadoGrueso / Fino | kg/m³ | Grava y arena |
| Edad | días | Curado antes del ensayo |
| **Resistencia** | **MPa** | **Variable a predecir (y)** |

Fuente: Prof. I-Cheng Yeh · 1 030 mezclas · regresión supervisada.

Detalle ampliado: [`data/DATOS.md`](data/DATOS.md).


## Pregunta 1 — Contexto del hormigón y Machine Learning

### 📘 Subpreguntas
- **1.a** ¿Predecir resistencia en MPa es **regresión** o **clasificación**?
- **1.b** ¿Por qué usamos MPa y no una etiqueta «bueno/malo»?

En planta, un modelo puede **explorar dosificaciones** antes de ensayos destructivos costosos.


#### ✍️ Tu respuesta (opcional, 2–3 líneas)




In [ ]:
### TU TAREA AQUÍ ###
# Indica el tipo de problema de ML para este lab.
TIPO_PROBLEMA = "regresion"  # Prueba cambiar a "clasificacion" y revisa la autoevaluación

print(f"Tipo de problema elegido: {TIPO_PROBLEMA}")


In [ ]:
# --- Autoevaluación 1 ---
r = verificar_tipo_problema(TIPO_PROBLEMA)
resumen_seccion('1 — Contexto ML', r)


## Pregunta 2 — Carga del dataset

### 📘 Subpreguntas
- **2.a** ¿Cuántas **features** hay frente a 1 **target**?
- **2.b** ¿Por qué convertimos el XLS UCI a CSV en español?


In [ ]:
# --- PRE-ESCRITO: carga desde data/concrete.csv ---
RUTA_DATOS = Path("data/concrete.csv")
df = pd.read_csv(RUTA_DATOS)
print(f"Archivo: {RUTA_DATOS} | Forma: {df.shape[0]} filas × {df.shape[1]} columnas")


In [ ]:
### TU TAREA AQUÍ ###
N_FILAS_HEAD = 5  # Prueba 3, 8 o 10

print(f"Primeras {N_FILAS_HEAD} mezclas:")
display(df.head(N_FILAS_HEAD))


In [ ]:
# --- Autoevaluación 2 ---
r = verificar_carga(df, N_FILAS_HEAD)
resumen_seccion('2 — Carga', r)
print("ℹ️ 8 features + 1 target (Resistencia).")


## Pregunta 3 — Calidad de datos

### 📘 Subpreguntas
- **3.a** ¿Hay valores nulos en el dataset?
- **3.b** Si fueras jefe de planta, ¿qué columna revisarías primero y por qué?


In [ ]:
### TU TAREA AQUÍ ###
COLUMNA_REVISAR = "Agua"  # Prueba "Edad" o "Cemento"

stats_col = df[COLUMNA_REVISAR].describe()
print(f"Estadísticas de «{COLUMNA_REVISAR}»:")
display(stats_col)


In [ ]:
# --- Autoevaluación 3 ---
r = verificar_columna(df, COLUMNA_REVISAR)
resumen_seccion('3 — Calidad', r)


## Pregunta 4 — Estadísticas descriptivas

### 📘 Subpreguntas
- **4.a** Mirando `describe()`, ¿qué variable tiene mayor dispersión relativa (std vs media)?
- **4.b** ¿Qué implicación tiene para el modelado?


In [ ]:
### TU TAREA AQUÍ ###
COLUMNAS_RESUMEN = ["Agua", "Edad", "Resistencia"]  # Quita o agrega columnas

resumen = df[COLUMNAS_RESUMEN].describe()
display(resumen)


In [ ]:
# --- Autoevaluación 4 ---
r = verificar_resumen(resumen, COLUMNAS_RESUMEN)
resumen_seccion('4 — Describe', r)


## Pregunta 5 — Distribución del target (Resistencia)

### 📘 Subpreguntas
- **5.a** ¿Hay más mezclas por debajo o por encima de 40 MPa?
- **5.b** ¿Qué sesgo visual ves en el histograma?


In [ ]:
# --- PRE-ESCRITO: histograma base ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['Resistencia'], bins=30, color='#3498db', edgecolor='white')
ax.set_xlabel('Resistencia (MPa)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de resistencia a compresión')
plt.tight_layout()
plt.show()


In [ ]:
### TU TAREA AQUÍ ###
UMBRAL_RESISTENCIA = 40  # MPa — experimenta: 30, 35, 45

n_fuertes = int((df['Resistencia'] >= UMBRAL_RESISTENCIA).sum())
n_debiles = int((df['Resistencia'] < UMBRAL_RESISTENCIA).sum())
print(f"≥ {UMBRAL_RESISTENCIA} MPa: {n_fuertes} | < {UMBRAL_RESISTENCIA} MPa: {n_debiles}")


In [ ]:
# --- Autoevaluación 5 ---
r = verificar_umbral_resistencia(df, UMBRAL_RESISTENCIA, n_fuertes, n_debiles)
resumen_seccion('5 — Target', r)


## Pregunta 6 — Correlación entre variables

### 📘 Subpreguntas
- **6.a** ¿Qué ingrediente correlaciona **más** con Resistencia (valor absoluto)?
- **6.b** ¿El signo de Agua coincide con la física del hormigón?


In [ ]:
# --- PRE-ESCRITO: matriz de correlación ---
corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlación — dosificación vs resistencia')
plt.tight_layout()
plt.show()


In [ ]:
### TU TAREA AQUÍ ###
TOP_N_CORR = 3  # Prueba 4 o 5

corr_target = corr['Resistencia'].drop('Resistencia')
top_corr = corr_target.abs().sort_values(ascending=False).head(TOP_N_CORR).index.tolist()
print("Top correlaciones con Resistencia (|r|):")
for nombre in top_corr:
    print(f"  {nombre}: r = {corr_target[nombre]:.3f}")


In [ ]:
# --- Autoevaluación 6 ---
from _verificar import verificar
r = verificar_correlaciones(top_corr, TOP_N_CORR)
r.append(
    verificar(
        corr_target['Agua'] < 0,
        "✅ Agua correlaciona negativamente con resistencia (física coherente).",
        "❌ Agua debería tener correlación negativa con resistencia.",
    )
)
resumen_seccion('6 — Correlación', r)


## Pregunta 7 — Relación física Agua vs Resistencia

### 📘 Subpreguntas
- **7.a** ¿Qué patrón ves en el scatter?
- **7.b** ¿Por qué más agua suele **bajar** la resistencia?


In [ ]:
# --- PRE-ESCRITO: scatter con filtro parametrizable ---
def graficar_agua_resistencia(datos, titulo_extra=''):
    fig, ax = plt.subplots(figsize=(8, 5))
    sc = ax.scatter(
        datos['Agua'], datos['Resistencia'],
        c=datos['Edad'], cmap='viridis', alpha=0.7, edgecolors='white', linewidth=0.3,
    )
    ax.set_xlabel('Agua (kg/m³)')
    ax.set_ylabel('Resistencia (MPa)')
    ax.set_title(f'Agua vs Resistencia {titulo_extra}')
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Edad (días)')
    plt.tight_layout()
    plt.show()


In [ ]:
### TU TAREA AQUÍ ###
EDAD_MIN_DIAS = 28  # Prueba 7, 14 o 56 (días de curado mínimo)

df_filtrado = df[df['Edad'] >= EDAD_MIN_DIAS]
n_filtradas = len(df_filtrado)
graficar_agua_resistencia(df_filtrado, f"(Edad ≥ {EDAD_MIN_DIAS} días, n={n_filtradas})")


In [ ]:
# --- Autoevaluación 7 ---
r = verificar_scatter_filtro(n_filtradas, EDAD_MIN_DIAS)
resumen_seccion('7 — Agua vs Resistencia', r)


## Pregunta 8 — Partición train / test

Puente desde Lab 0: `X` = features, `y` = Resistencia, luego `model.fit(X_train, y_train)`.

### 📘 Subpreguntas
- **8.a** ¿Para qué sirve `random_state=42`?
- **8.b** ¿Por qué no evaluamos el modelo con los mismos datos de entrenamiento?


In [ ]:
# --- PRE-ESCRITO: definir X e y ---
TARGET = "Resistencia"
FEATURES_TODAS = [c for c in df.columns if c != TARGET]
X = df[FEATURES_TODAS]
y = df[TARGET]
print(f"X: {X.shape} | y: {y.shape}")


In [ ]:
### TU TAREA AQUÍ ###
TEST_SIZE = 0.2       # Prueba 0.15 o 0.25
RANDOM_STATE = 42     # Cambia y observa que varían las métricas

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")


In [ ]:
# --- Autoevaluación 8 ---
r = verificar_split(len(X_train), len(X_test), TEST_SIZE, RANDOM_STATE)
resumen_seccion('8 — Train/Test', r)


## Pregunta 9 — Random Forest (hiperparámetros)

### 📘 Subpreguntas
- **9.a** ¿Qué controla `n_estimators`?
- **9.b** ¿Qué pasa con R² si quitas `Edad` de `COLUMNAS_X`?


In [ ]:
### TU TAREA AQUÍ ###
N_ESTIMATORS = 100    # Prueba 50 o 200
MAX_DEPTH = 10        # Prueba 5 o None
COLUMNAS_X = FEATURES_TODAS  # Quita 'Edad' y vuelve a ejecutar

X_train_sel = X_train[COLUMNAS_X]
X_test_sel = X_test[COLUMNAS_X]

modelo = RandomForestRegressor(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    random_state=RANDOM_STATE,
)
modelo.fit(X_train_sel, y_train)

y_pred = modelo.predict(X_test_sel)
r2_test = r2_score(y_test, y_pred)

importancias = dict(zip(COLUMNAS_X, modelo.feature_importances_))
print(f"R² test = {r2_test:.3f} | Features usadas: {COLUMNAS_X}")


In [ ]:
# --- Autoevaluación 9 ---
MIN_R2 = 0.75 if 'Edad' in COLUMNAS_X else 0.30
top_esperada = 'Edad' if 'Edad' in COLUMNAS_X else max(importancias, key=importancias.get)
r = verificar_modelo(r2_test, MIN_R2, importancias, top_esperada)
resumen_seccion('9 — Random Forest', r)
print(f"ℹ️ R² sin Edad cae fuerte (~0.39): el curado explica mucha variabilidad.")


## Pregunta 10 — Feature Importance y validación visual

### 📘 Subpreguntas
- **10.a** ¿Qué variable impacta más según el bosque aleatorio?
- **10.b** ¿Cómo usarías esto para **optimizar dosificaciones** en obra?


In [ ]:
### TU TAREA AQUÍ ###
N_TOP_IMPORTANCIAS = 5  # Prueba 3 u 8

serie_imp = pd.Series(importancias).sort_values(ascending=False)
top_importancias = serie_imp.head(N_TOP_IMPORTANCIAS)
importancias_ordenadas = top_importancias.index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_importancias.iloc[::-1].plot(kind='barh', ax=axes[0], color='#2ecc71')
axes[0].set_title(f'Top {N_TOP_IMPORTANCIAS} — Feature Importance')
axes[0].set_xlabel('Importancia relativa')

axes[1].scatter(y_test, y_pred, alpha=0.6, edgecolors='white')
lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[1].plot(lim, lim, 'r--', label='Predicción perfecta')
axes[1].set_xlabel('Resistencia real (MPa)')
axes[1].set_ylabel('Resistencia predicha (MPa)')
axes[1].set_title(f'Validación visual (R² = {r2_test:.3f})')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Top importancias:", importancias_ordenadas)


In [ ]:
# --- Autoevaluación 10 ---
r = verificar_importancias(importancias_ordenadas, N_TOP_IMPORTANCIAS)
r.append(r2_test >= 0.75 if 'Edad' in COLUMNAS_X else r2_test >= 0.30)
if r[-1]:
    print(f"✅ Gráfico final coherente (R² = {r2_test:.3f}).")
else:
    print("❌ Revisa COLUMNAS_X y hiperparámetros antes del cierre.")
resumen_seccion('10 — Importancia y gráfico final', r)


## Cierre profesional

### ✍️ Reflexión final (3 frases)
- La variable que más impactó fue ___ porque ___.
- Para reducir costos en planta usaría el modelo para ___.
- Antes de obra real validaría con ___.

---
**Checklist:** 10 autoevaluaciones en ✅ · gráficos revisados · [`data/DATOS.md`](data/DATOS.md) leído.
